In [1]:
!pip install transformers torch

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 1. Model Loading
# We are using 'microsoft/DialoGPT-medium' for a balance between speed and quality.
model_name = "microsoft/DialoGPT-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

def start_chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today? (Type 'exit' or 'quit' to stop)")

    # Initialize chat history
    chat_history_ids = None

    while True:
        # 2. User Input Handling
        user_input = input("User: ")

        # 5. Exit Condition
        if user_input.lower() in ['exit', 'quit']:
            print("Chatbot: Goodbye! Have a great day.")
            break

        # 3. Response Generation (Encoding input)
        # We encode the new user input and add the EOS (End of Sentence) token
        new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

        # 4. Continuous Conversation (Appending to history)
        # If it's the first turn, history is just the input.
        # Otherwise, we append the new input to the previous conversation tokens.
        bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1) if chat_history_ids is not None else new_user_input_ids

        # Generate a response
        # max_length defines how long the total conversation can be
        # pad_token_id is set to eos_token_id for open-ended generation
        chat_history_ids = model.generate(
            bot_input_ids,
            max_length=1000,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3,       # Prevents the bot from repeating phrases
            do_sample=True,               # Use sampling for more natural variety
            top_k=100,
            top_p=0.7,
            temperature=0.8
        )

        # Decode the last output tokens from the model
        response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

        print(f"Chatbot: {response}")

if __name__ == "__main__":
    start_chatbot()

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chatbot: Hello! I am your AI assistant. How can I help you today? (Type 'exit' or 'quit' to stop)
User: hi
Chatbot: Hello!
User: what is AI?
Chatbot: AI Artificial Intelligence
User: who created python?
Chatbot: I think you mean python?
User: yes
Chatbot: You're welcome
User: quit
Chatbot: Goodbye! Have a great day.
